In [60]:

import kagglehub
import pandas as pd
import numpy as np
import os

In [61]:
path = kagglehub.dataset_download("xfkzujqjvx97n/football-datasets")
tables = {}
print("Path to dataset files:", path)
for root, dirs, files in os.walk(path):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), path))
        if f.endswith(".csv"):
            nom = os.path.splitext(f)[0]
            tables[nom] = pd.read_csv(os.path.join(root, f), low_memory=False)


Path to dataset files: /Users/charles/.cache/kagglehub/datasets/xfkzujqjvx97n/football-datasets/versions/2
player_profiles/player_profiles.csv
player_national_performances/player_national_performances.csv
player_performances/player_performances.csv
player_injuries/player_injuries.csv
team_children/team_children.csv
team_competitions_seasons/team_competitions_seasons.csv
player_latest_market_value/player_latest_market_value.csv
player_market_value/player_market_value.csv
player_teammates_played_with/player_teammates_played_with.csv
team_details/team_details.csv
transfer_history/transfer_history.csv


In [62]:
for nom, df in tables.items():
    print(f"\n=== {nom} ===")
    print(df.shape)
    print(df.columns.tolist())


=== player_profiles ===
(92671, 34)
['player_id', 'player_slug', 'player_name', 'player_image_url', 'name_in_home_country', 'date_of_birth', 'place_of_birth', 'country_of_birth', 'height', 'citizenship', 'is_eu', 'position', 'main_position', 'foot', 'current_club_id', 'current_club_name', 'joined', 'contract_expires', 'outfitter', 'social_media_url', 'player_agent_id', 'player_agent_name', 'contract_option', 'date_of_last_contract_extension', 'on_loan_from_club_id', 'on_loan_from_club_name', 'contract_there_expires', 'second_club_url', 'second_club_name', 'third_club_url', 'third_club_name', 'fourth_club_url', 'fourth_club_name', 'date_of_death']

=== player_national_performances ===
(92701, 9)
['player_id', 'team_id', 'matches', 'goals', 'shirt_number', 'debut', 'coach_id', 'debut_game_id', 'career_state']

=== player_performances ===
(1878719, 20)
['player_id', 'season_name', 'competition_id', 'competition_name', 'team_id', 'team_name', 'nb_in_group', 'nb_on_pitch', 'goals', 'assist

In [63]:
# Test de l'unicité des clés primaires
for t, k in [("player_profiles", "player_id"), ("team_details", "club_id"),
             ("player_latest_market_value", "player_id")]:
    print(f"{t}.{k} unique :", tables[t][k].is_unique)

player_profiles.player_id unique : True
team_details.club_id unique : True
player_latest_market_value.player_id unique : True


In [64]:
def check_lien(src, col, ref, ref_col):
    s, r = tables[src][col].dropna(), tables[ref][ref_col].dropna()
    return {"lien": f"{src}.{col} → {ref}.{ref_col}",
            "% distincts": round(pd.Series(s.unique()).isin(r).mean() * 100, 1),
            "% lignes": round(s.isin(r).mean() * 100, 1)}
pd.set_option("display.max_colwidth", None)
liens = [
    # Couverture : part des joueurs de player_profiles présents dans chaque table
    ("player_profiles", "player_id", "player_performances", "player_id"),
    ("player_profiles", "player_id", "transfer_history", "player_id"),
    ("player_profiles", "player_id", "player_market_value", "player_id"),
    ("player_profiles", "player_id", "player_latest_market_value", "player_id"),
    ("player_profiles", "player_id", "player_injuries", "player_id"),
    ("player_injuries", "player_id", "player_profiles", "player_id"),
    ("player_profiles", "player_id", "player_national_performances", "player_id"),
    ("player_profiles", "player_id", "player_teammates_played_with", "player_id"),
    ("player_profiles", "player_id", "player_teammates_played_with", "teammate_player_id"),
    # Liens vers les clubs 
    ("player_performances", "team_id", "team_details", "club_id"),
    ("transfer_history", "from_team_id", "team_details", "club_id"),
    ("transfer_history", "to_team_id", "team_details", "club_id"),
    ("player_profiles", "current_club_id", "team_details", "club_id"),
    ("team_children", "parent_team_id", "team_details", "club_id"),
    ("team_children", "child_team_id", "team_details", "club_id"),
    ("player_national_performances", "team_id", "team_details", "club_id"),
    ("team_competitions_seasons", "club_id", "team_details", "club_id"),
    # Liens vers les compétitions (sens inchangé)
    ("player_performances", "competition_id", "team_competitions_seasons", "competition_id"),
    ("team_details", "competition_id", "team_competitions_seasons", "competition_id"),
]
display(pd.DataFrame([check_lien(*l) for l in liens]))

,lien,% distincts,% lignes
0,player_profiles.player_id → player_performances.player_id,95.4,95.4
1,player_profiles.player_id → transfer_history.player_id,98.4,98.4
2,player_profiles.player_id → player_market_value.player_id,74.9,74.9
3,player_profiles.player_id → player_latest_market_value.player_id,74.9,74.9
4,player_profiles.player_id → player_injuries.player_id,37.3,37.3
5,player_injuries.player_id → player_profiles.player_id,100.0,100.0
6,player_profiles.player_id → player_national_performances.player_id,30.4,30.4
7,player_profiles.player_id → player_teammates_played_with.player_id,91.1,91.1
8,player_profiles.player_id → player_teammates_played_with.teammate_player_id,56.9,56.9
9,player_performances.team_id → team_details.club_id,16.0,76.3


**Le noyeau des données de base dont on aura besoin plus tard est solide.** Les quatres tables indispensables (`player_profiles`, `player_performances`, `player_injuries`, `player_market_value`) sont reliées par player_id. Toutes les lignes des tables du noyau correspondent à un joueur connu, mais la couverture des joueurs varie : 95 % ont des performances, 75 % une valeur marchande et 37 % au moins une blessure.

**Les faiblesses portent sur les clubs.** Ce sont plus des variables de contexte.

Les liens entre les tables `team_details` et `team_competitions_seasons` est établie à 100% dans les deux sens. Cependant les liens depuis les autres tables sont plus faibles.

player_performances (76%) :  table clé pour attribuer à chaque (joueur, saison) un club. À voir si les 24% manquants concernent de petits club.

transfer_history (48 ~ 50 %) : lien faible mais pas bloquant car le modèle de perte devrait reposer sur d’autres variables que les équipes.

player_profiles, team_children et player_national_performances : inutile ou usage secondaire pour les modélisation.

player_profiles.current_team_id : inutile pour une base historique 



In [65]:
joueurs = tables["player_profiles"]["player_id"]

tables_infos_joueurs = {
    "player_performances": "player_id",
    "player_market_value": "player_id",
    "player_latest_market_value": "player_id",
    "player_injuries": "player_id",
    "transfer_history": "player_id",
    "player_national_performances": "player_id",
    "player_teammates_played_with": "player_id",
}

couverture = pd.DataFrame({
    "table": list(tables_infos_joueurs),
    "% joueurs couverts": [round(joueurs.isin(tables[t][c]).mean() * 100, 1)
                           for t, c in tables_infos_joueurs.items()],
}).sort_values("% joueurs couverts", ascending=False)
display(couverture)

# Couverture combinée des infos indispensables au sujet
a_perf  = joueurs.isin(tables["player_performances"]["player_id"])
a_value = joueurs.isin(tables["player_market_value"]["player_id"])
a_inj   = joueurs.isin(tables["player_injuries"]["player_id"])

print("Performances + valeur marchande :", round((a_perf & a_value).mean() * 100, 1), "%")
print("Performances + valeur + au moins une blessure :", round((a_perf & a_value & a_inj).mean() * 100, 1), "%")

,table,% joueurs couverts
4,transfer_history,98.4
0,player_performances,95.4
6,player_teammates_played_with,91.1
1,player_market_value,74.9
2,player_latest_market_value,74.9
3,player_injuries,37.3
5,player_national_performances,30.4


Performances + valeur marchande : 74.6 %
Performances + valeur + au moins une blessure : 36.4 %


In [66]:
noms = (tables["player_performances"][["team_id", "team_name"]]
        .drop_duplicates("team_id").set_index("team_id")["team_name"])

top_nat = tables["player_national_performances"]["team_id"].value_counts().head(10)
print(top_nat.to_frame("nb_lignes").join(noms))

         nb_lignes team_name
team_id                     
21746          839       NaN
21100          760       NaN
21426          754       NaN
23133          728       NaN
16371          681       NaN
23243          667       NaN
9067           649       NaN
5710           648       NaN
16373          643       NaN
31269          606       NaN


Aucune equipe de player_national_performances n'est référencée dans TEAM_DETAILS car il s'agit d'une équipe nationale et non d'une équipe de club.

### Intégrité des données :

- `player_profiles` : 100 % des player_id sont uniques et non nuls. Les variables utiles à la modélisation sont quasi complète : id, name, main_position, date_of_birth, nationality, current_club_id, current_club_name. D'autres variables plus contextuelles sont très imcompletes mais non necessaires à la modélisation.

- `team_details` : 100 % des team_id sont uniques et non nuls. L'identité des clubs (nom, pays) est toujours renseignée. Les informations de compétition (competition_id, nom, club_division) manquent conjointement pour 23 % des clubs. Comme la table ne donne qu’une seule ligne par club, le championnat par saison devra de toute façon être repris de `team_competitions_seasons`.

- `team_children` : La table est complète sans aucune valeur manquante. Cependant son usage reste secondaire : rattacher les équipes réserves à leur club parent.

- `team_competitions_seasons` : Les clés club_id et season_id sont complètes. Les informations de compétition manquent conjointement pour 16 % des couples club-saison. C’est la table de référence pour attribuer un championnat et une division à chaque club par saison.


### `player_profiles`

In [67]:
print("Proportion de valeurs manquantes par variables dans la table Player_profiles")
pd.DataFrame({
    "nb" : tables["player_profiles"].isna().sum(),
    "pct" : (tables["player_profiles"].isna().mean() * 100 ).round(2)
}).sort_values("pct")


Proportion de valeurs manquantes par variables dans la table Player_profiles


,nb,pct
player_id,0,0.00
player_slug,0,0.00
player_image_url,0,0.00
current_club_name,0,0.00
current_club_id,0,0.00
main_position,2,0.00
height,4,0.00
is_eu,0,0.00
position,2,0.00
citizenship,57,0.06


In [68]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_profiles"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_profiles"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_profiles"].columns[tables["player_profiles"].isna().all()])


Nombre de lignes avec au moins une valeurs manquante :  92671
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


#### Doublons

In [69]:
cols_hors_id = [c for c in tables["player_profiles"].columns if c not in ("player_id", "player_name", "player_image_url")]

print("Doublons stricts            :", tables["player_profiles"].duplicated().sum())
print("player_id répétés           :", tables["player_profiles"]["player_id"].duplicated().sum())
print("Doublons hors identifiant   :", tables["player_profiles"].duplicated(cols_hors_id).sum())
print("Lignes au slug répété       :", tables["player_profiles"]["player_slug"].duplicated(keep=False).sum())

cle_identite = ["player_slug", "date_of_birth", "citizenship"]
suspects = tables["player_profiles"][tables["player_profiles"].duplicated(cle_identite, keep=False) & tables["player_profiles"]["date_of_birth"].notna()]
print("Lignes même slug + naissance + nationalité :", len(suspects))
display(suspects.sort_values(cle_identite)[
    ["player_id", "player_slug", "date_of_birth", "citizenship", "height",
     "position", "current_club_name", "joined"]])

Doublons stricts            : 0
player_id répétés           : 0
Doublons hors identifiant   : 0
Lignes au slug répété       : 7372
Lignes même slug + naissance + nationalité : 4


,player_id,player_slug,date_of_birth,citizenship,height,position,current_club_name,joined
7825,1096879,felipe-caua,2004-05-06,Brazil,0.0,Defender - Centre-Back,Sampaio Corrêa FC (MA) U20,NaN
9923,1370267,felipe-caua,2004-05-06,Brazil,0.0,Defender - Centre-Back,Sampaio Corrêa FC (MA),NaN
24230,179078,pacheta,1981-09-09,Spain,185.0,Attack - Centre-Forward,Retired,2021-07-01
54946,505313,pacheta,1981-09-09,Spain,0.0,Attack,Retired,NaN


Il semble n'y avoir que deux doublons strictes, car même nom, date de naissance et nationalité. On verra au moment de la joiture la décision à prendre car il ne faudrait pas supprimer un player_id avec des lignes dans les autres tables.

In [86]:
tables["player_profiles"]["suspected_duplicate"] = tables["player_profiles"]["player_id"].isin(suspects["player_id"])

#### Selection des colonnes utiles au sujet 

In [70]:
suppressions = {
    "Hors sujet (aucun lien avec le risque de blessure, de décès ou la valeur)": [
        "player_image_url", "outfitter", "social_media_url",
        "player_agent_id", "player_agent_name"],
    "Redondant ou trop détaillé (name_in_home_country doublonne le nom ; country_of_birth suffit)": [
        "name_in_home_country", "place_of_birth"],
    "Situation actuelle, pas historique : le club par saison vient de player_performances": [
        "joined", "contract_expires", "contract_option", "date_of_last_contract_extension",
        "on_loan_from_club_id", "on_loan_from_club_name", "contract_there_expires"],
    "Plus de 98 % de valeurs manquantes": [
        "second_club_url", "second_club_name", "third_club_url", "third_club_name",
        "fourth_club_url", "fourth_club_name"],
}

for i, (raison, cols) in enumerate(suppressions.items(), start=3):
    try:
        taux = (tables["player_profiles"][cols].isna().mean() * 100).round(1)
        print(f"{raison}\n  {dict(taux)}\n")
    except:
        pass
try:
    tables["player_profiles"] = tables["player_profiles"].drop(columns=[c for cols in suppressions.values() for c in cols])
except:
    pass
print(tables["player_profiles"].shape)

Hors sujet (aucun lien avec le risque de blessure, de décès ou la valeur)
  {'player_image_url': np.float64(0.0), 'outfitter': np.float64(93.6), 'social_media_url': np.float64(78.1), 'player_agent_id': np.float64(58.9), 'player_agent_name': np.float64(53.7)}

Redondant ou trop détaillé (name_in_home_country doublonne le nom ; country_of_birth suffit)
  {'name_in_home_country': np.float64(46.4), 'place_of_birth': np.float64(17.9)}

Situation actuelle, pas historique : le club par saison vient de player_performances
  {'joined': np.float64(3.6), 'contract_expires': np.float64(58.3), 'contract_option': np.float64(96.1), 'date_of_last_contract_extension': np.float64(91.0), 'on_loan_from_club_id': np.float64(95.9), 'on_loan_from_club_name': np.float64(95.9), 'contract_there_expires': np.float64(97.3)}

Plus de 98 % de valeurs manquantes
  {'second_club_url': np.float64(98.9), 'second_club_name': np.float64(98.9), 'third_club_url': np.float64(100.0), 'third_club_name': np.float64(100.0), 'fo

#### Nom du joueur

`player_name` contient l'identifiant entre parenthèses (« Miroslav Klose (10) »). On retire ce suffixe. Les noms manquants sont reconstruits à partir du slug.

In [71]:
motif_suffixe = r"\s*\(\d+\)$"
avec_suffixe = tables["player_profiles"]["player_name"].str.contains(motif_suffixe, na=False)
tables["player_profiles"]["player_name"] = tables["player_profiles"]["player_name"].str.replace(motif_suffixe, "", regex=True).str.strip()

manquant = tables["player_profiles"]["player_name"].isna()
tables["player_profiles"].loc[manquant, "player_name"] = tables["player_profiles"].loc[manquant, "player_slug"].str.replace("-", " ").str.title()

print("Suffixes retirés :", avec_suffixe.sum(), "| noms reconstruits :", manquant.sum())

Suffixes retirés : 92151 | noms reconstruits : 520


#### Date de naissance 


le format actuel est `YYYY-MM-DD`. On peut le convertir en datetime pour calculer l'âge du joueur à chaque saison. Il n'y a pas de date de naissance manquante.

In [72]:
tables["player_profiles"]["date_of_birth"] = pd.to_datetime(tables["player_profiles"]["date_of_birth"], format="%Y-%m-%d", errors="coerce")
erreurs = tables["player_profiles"]["date_of_birth"].isna().sum() - tables["player_profiles"]["date_of_birth"].isna().sum()
print("Valeurs non convertibles :", erreurs)
print("Manquantes :", tables["player_profiles"]["date_of_birth"].isna().sum())
print("Plage :", tables["player_profiles"]["date_of_birth"].min().date(), "→", tables["player_profiles"]["date_of_birth"].max().date())


Valeurs non convertibles : 0
Manquantes : 1006
Plage : 1952-02-13 → 2010-10-12


#### Date de décès

De même on convertit le format en datetime.

In [73]:
motif_deces = r"^(?P<date>\d{2}\.\d{2}\.\d{4}) \((?P<age>\d+)\)$"
extrait = tables["player_profiles"]["date_of_death"].str.extract(motif_deces)
assert extrait["date"].notna().sum() == tables["player_profiles"]["date_of_death"].notna().sum(), "format inattendu"

tables["player_profiles"]["date_of_death"] = pd.to_datetime(extrait["date"], format="%d.%m.%Y")
tables["player_profiles"]["is_dead"] = tables["player_profiles"]["date_of_death"].notna()

# Controle de la concordance des dates
d, n = tables["player_profiles"]["date_of_death"], tables["player_profiles"]["date_of_birth"]
anniv_pas_passe = (d.dt.month < n.dt.month) | ((d.dt.month == n.dt.month) & (d.dt.day < n.dt.day))
age_calcule = d.dt.year - n.dt.year - anniv_pas_passe.astype(int)
concordance = (age_calcule[tables["player_profiles"]["is_dead"]] == extrait["age"][tables["player_profiles"]["is_dead"]].astype(int)).mean()

print("Décès :", tables["player_profiles"]["is_dead"].sum())
print("Plage :", tables["player_profiles"]["date_of_death"].min().date(), "→", tables["player_profiles"]["date_of_death"].max().date())
print(f"Concordance âge indiqué / recalculé : {concordance:.0%}")

Décès : 272
Plage : 2001-08-19 → 2025-08-14
Concordance âge indiqué / recalculé : 100%


#### Statut actuel du joueur





In [74]:
print(tables["player_profiles"]["current_club_id"].value_counts().head(10))
print(tables["player_profiles"]["current_club_name"].value_counts().head(10))

current_club_id
123       23380
515        8599
75         1930
4023        271
2113        102
26370        47
20406        45
21459        44
112192       44
86387        41
Name: count, dtype: int64
current_club_name
Retired                   23380
Without Club               8599
Unknown                    1930
---                         271
Career break                102
Arsenal FC                   58
Botafogo FC                  53
Pro Vercelli Primavera       47
Reggiana Primavera           45
Gangwon FC                   44
Name: count, dtype: int64


In [75]:
pseudo_clubs = (tables["player_profiles"].groupby("current_club_id")
                .agg(libelle=("current_club_name", "unique"),
                     nb_joueurs=("player_id", "size"),
                     nb_decedes=("date_of_death", lambda s: s.notna().sum()))
                .sort_values("nb_joueurs", ascending=False)
                .head(8))
pseudo_clubs

,libelle,nb_joueurs,nb_decedes
current_club_id,,,
123,[Retired],23380,3
515,[Without Club],8599,0
75,[Unknown],1930,1
4023,[---],271,268
2113,[Career break],102,0
26370,[Pro Vercelli Primavera],47,0
20406,[Reggiana Primavera],45,0
21459,[Gangwon FC],44,0


`current_club_id` ne sert pas comme club historique, mais certains identifiants sont des codes de statut : 123 = *Retired*, 515 = *Without Club*, 2113 = *Career break*, 75 = *Unknown*, et 4023 = `---`, qui correspond aux joueurs décédés. On en tire une variable `current_status`, utile pour savoir si un joueur est encore en activité (fin d'exposition au risque), puis on retire les deux colonnes de club.

In [76]:
print(pd.crosstab(tables["player_profiles"]["current_club_id"].eq(4023), tables["player_profiles"]["is_dead"],
                  rownames=["club « --- »"], colnames=["is_dead"]))

codes_statut = {123: "retired", 515: "without_club", 2113: "career_break", 75: "unknown", 4023: "unknown"}
tables["player_profiles"]["current_status"] = tables["player_profiles"]["current_club_id"].map(codes_statut).fillna("active")
tables["player_profiles"].loc[tables["player_profiles"]["is_dead"], "current_status"] = "deceased"
print(tables["player_profiles"]["current_status"].value_counts())

tables["player_profiles"] = tables["player_profiles"].drop(columns=["current_club_id", "current_club_name"])


is_dead       False  True 
club « --- »              
False         92396      4
True              3    268
current_status
active          58389
retired         23377
without_club     8599
unknown          1932
deceased          272
career_break      102
Name: count, dtype: int64


#### Taille

toutes les tailles en dessous de 1,40 m sont suspectes pour de joueurs de foot professionnels. On les considères comme des valeurs manquantes.

In [79]:
print("height = 0 :", (tables["player_profiles"]["height"] == 0).sum(), "| 0 < height < 140 :", tables["player_profiles"]["height"].between(0.1, 139).sum())
aberrant = tables["player_profiles"]["height"] < 140
tables["player_profiles"].loc[aberrant, "height"] = np.nan
print("Plage après :", tables["player_profiles"]["height"].min(), "→", tables["player_profiles"]["height"].max(),
      f"| manquants : {tables["player_profiles"]['height'].isna().mean():.1%}")

height = 0 : 18867 | 0 < height < 140 : 1
Plage après : 152.0 → 210.0 | manquants : 20.4%


#### Nationalité

In [81]:
tables["player_profiles"]["citizenship"] = tables["player_profiles"]["citizenship"].str.replace("Cookinseln", "Cook Islands")
nat = tables["player_profiles"]["citizenship"].str.split(r"\s{2,}", expand=True)
print("Nombre maximal de nationalités :", nat.shape[1])

tables["player_profiles"]["nationality_1"] = nat[0].str.strip()
tables["player_profiles"]["nationality_2"] = nat[1].str.strip()
tables["player_profiles"]["has_dual_nationality"] = tables["player_profiles"]["nationality_2"].notna()
print("Doubles nationalités :", tables["player_profiles"]["has_dual_nationality"].sum())
tables["player_profiles"] = tables["player_profiles"].drop(columns="citizenship")

Nombre maximal de nationalités : 2
Doubles nationalités : 19956


#### Poste

`position` combine le poste général et le poste détaillé (« Defender - Right-Back »). La partie avant le tiret est identique à `main_position` : on ne garde donc que la partie détaillée dans `sub_position`. Pour les joueurs dont seul le poste général est connu (« Mitables["player_profiles"]ield »), `sub_position` reste vide. Le poste détaillé peut compter pour le risque de blessure (un latéral ne court pas comme un défenseur central).

In [82]:
poste = tables["player_profiles"]["position"].str.split(" - ", n=1, expand=True)
assert (poste[0].eq(tables["player_profiles"]["main_position"]) | tables["player_profiles"]["position"].isna()).all()

tables["player_profiles"]["sub_position"] = poste[1]
tables["player_profiles"].loc[tables["player_profiles"]["main_position"] == "Goalkeeper", "sub_position"] = "Goalkeeper"
generique = tables["player_profiles"]["sub_position"].isna() & tables["player_profiles"]["main_position"].notna()
print("Postes sans détail :", generique.sum())
display(pd.crosstab(tables["player_profiles"]["sub_position"].fillna("(non précisé)"), tables["player_profiles"]["main_position"]))

tables["player_profiles"] = tables["player_profiles"].drop(columns="position")

Postes sans détail : 7046


main_position,Attack,Defender,Goalkeeper,Midfield
sub_position,,,,
(non précisé),1885,2013,0,3148
Attacking Midfield,0,0,0,5821
Central Midfield,0,0,0,9546
Centre-Back,0,15074,0,0
Centre-Forward,12070,0,0,0
Defensive Midfield,0,0,0,6999
Goalkeeper,0,0,10276,0
Left Midfield,0,0,0,1314
Left Winger,4756,0,0,0


#### Pied fort


In [83]:
tables["player_profiles"]["foot"] = tables["player_profiles"]["foot"].str.strip().str.lower()
print(tables["player_profiles"]["foot"].value_counts(dropna=False))

foot
right    49186
NaN      23488
left     16870
both      3127
Name: count, dtype: int64


#### Ajencement des variables et controles finaux 

In [87]:
colonnes_finales = [
    "player_id", "player_name", "player_slug",
    "date_of_birth", "country_of_birth", "nationality_1", "nationality_2",
    "has_dual_nationality", "is_eu",
    "height", "foot", "main_position", "sub_position",
    "date_of_death", "is_dead", "current_status", "suspected_duplicate",
]
assert set(colonnes_finales) == set(tables["player_profiles"].columns), set(tables["player_profiles"].columns) ^ set(colonnes_finales)
propre_profiles = tables["player_profiles"][colonnes_finales].copy()

for c in ["foot", "main_position", "sub_position", "current_status"]:
    propre_profiles[c] = propre_profiles[c].astype("category")

propre_profiles.dtypes

player_id                        int64
player_name                        str
player_slug                        str
date_of_birth           datetime64[us]
country_of_birth                   str
nationality_1                      str
nationality_2                      str
has_dual_nationality              bool
is_eu                             bool
height                         float64
foot                          category
main_position                 category
sub_position                  category
date_of_death           datetime64[us]
is_dead                           bool
current_status                category
suspected_duplicate               bool
dtype: object

In [88]:
assert propre_profiles["player_id"].is_unique and propre_profiles["player_id"].notna().all()
assert propre_profiles["date_of_birth"].dropna().between("1950-01-01", "2012-12-31").all()
assert propre_profiles["height"].dropna().between(140, 215).all()
assert (propre_profiles["is_dead"] == propre_profiles["date_of_death"].notna()).all()
assert (propre_profiles["date_of_death"].dropna() > propre_profiles.loc[propre_profiles["is_dead"], "date_of_birth"]).all()
assert (propre_profiles["has_dual_nationality"] == propre_profiles["nationality_2"].notna()).all()
assert not propre_profiles["player_name"].str.contains(r"\(\d+\)$").any()
print("Tous les contrôles passent.")

resume = pd.DataFrame({
    "type": propre_profiles.dtypes.astype(str),
    "% manquants": (propre_profiles.isna().mean() * 100).round(2),
    "nb distincts": propre_profiles.nunique(),
})
print(f"{tables["player_profiles"].shape} → {propre_profiles.shape}")

Tous les contrôles passent.
(92671, 17) → (92671, 17)


### `team_details`

In [ ]:
print(tables["team_details"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_details")
pd.DataFrame({
    "nb" : tables["team_details"].isna().sum(),
    "pct" : (tables["team_details"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(2175, 12)
Proportion de valeurs manquantes par variables dans la table team_details


,nb,pct
club_id,0,0.00
club_slug,0,0.00
club_name,0,0.00
logo_url,0,0.00
country_name,0,0.00
season_id,0,0.00
source_url,0,0.00
_last_modified_at,0,0.00
competition_id,506,23.26
competition_slug,506,23.26


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_details"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_details"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_details"].columns[tables["team_details"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  506
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_children`

In [ ]:
print(tables["team_children"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_children")
pd.DataFrame({
    "nb" : tables["team_children"].isna().sum(),
    "pct" : (tables["team_children"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(7695, 5)
Proportion de valeurs manquantes par variables dans la table team_children


,nb,pct
parent_team_id,0,0.0
parent_team_name,0,0.0
child_team_id,0,0.0
child_team_name,0,0.0
_last_modified_at,0,0.0


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_children"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_children"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_children"].columns[tables["team_children"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  0
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_competitions_seasons`

In [ ]:
print(tables["team_competitions_seasons"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_competitions_seasons")
pd.DataFrame({
    "nb" : tables["team_competitions_seasons"].isna().sum(),
    "pct" : (tables["team_competitions_seasons"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(196378, 7)
Proportion de valeurs manquantes par variables dans la table team_competitions_seasons


,nb,pct
club_id,0,0.00
team_name,0,0.00
season_id,0,0.00
_last_modified_at,0,0.00
competition_name,32189,16.39
competition_id,32189,16.39
club_division,32189,16.39


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_competitions_seasons"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_competitions_seasons"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_competitions_seasons"].columns[tables["team_competitions_seasons"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  32189
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `PLAYER_TEAMMATES_PLAYED_WITH`

In [ ]:
print(tables["player_teammates_played_with"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_teammates_played_with")
pd.DataFrame({
    "nb" : tables["player_teammates_played_with"].isna().sum(),
    "pct" : (tables["player_teammates_played_with"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1257342, 6)
Proportion de valeurs manquantes par variables dans la table player_teammates_played_with


,nb,pct
player_id,0,0.00
teammate_player_id,0,0.00
teammate_player_name,0,0.00
ppg_played_with,37080,2.95
minutes_played_with,804153,63.96
joint_goal_participation,1035177,82.33


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_teammates_played_with"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_teammates_played_with"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_teammates_played_with"].columns[tables["player_teammates_played_with"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1232285
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.


In [ ]:
print("Doublons stricts :", tables["player_teammates_played_with"].duplicated().sum())

Doublons stricts : 0


Dans player_teammates_played_with : le couple (player_id, teammate_id) est-il unique, ou un même coéquipier apparaît-il plusieurs fois (par exemple s'ils ont joué ensemble dans plusieurs clubs différents) ?

In [ ]:
nb_doublons_paires = (
    tables["player_teammates_played_with"]
    .duplicated(subset=["player_id", "teammate_player_id"])
    .sum()
)
print("Nombre de paires (joueur, coéquipier) répétées :", nb_doublons_paires)

Nombre de paires (joueur, coéquipier) répétées : 0


### `PLAYER INJURY HISTORIES`

In [ ]:
print(tables["player_injuries"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_injuries")
pd.DataFrame({
    "nb" : tables["player_injuries"].isna().sum(),
    "pct" : (tables["player_injuries"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(143195, 7)
Proportion de valeurs manquantes par variables dans la table player_injuries


,nb,pct
player_id,0,0.00
season_name,0,0.00
injury_reason,0,0.00
games_missed,0,0.00
from_date,22,0.02
days_missed,22,0.02
end_date,1523,1.06


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_injuries"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_injuries"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_injuries"].columns[tables["player_injuries"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1545
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.


HYPER IMPORTANT DE S'ATTARDER LÀ DESSUS

In [ ]:
print("Doublons stricts :", tables["player_injuries"].duplicated().sum())

Doublons stricts : 111


In [ ]:
lignes_par_joueur = tables["player_injuries"]["player_id"].value_counts()
print("Statistiques du nombre de blessures par joueur :")
print(lignes_par_joueur.describe())

Statistiques du nombre de blessures par joueur :
count    34561.000000
mean         4.143254
std          3.954910
min          1.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         15.000000
Name: count, dtype: float64


days_missed et games_missed ont-ils des valeurs négatives impossibles ?


In [ ]:
print(tables["player_injuries"][["days_missed", "games_missed"]].describe())
print()
print(tables["player_injuries"]["injury_reason"].value_counts().head(10))

         days_missed   games_missed
count  143173.000000  143195.000000
mean       51.719291       6.628479
std       104.437179       9.468380
min         1.000000       0.000000
25%        10.000000       1.000000
50%        22.000000       3.000000
75%        53.000000       7.000000
max      8655.000000     208.000000

injury_reason
unknown injury            27028
Muscle injury              6433
Hamstring injury           5786
Knee injury                5544
muscular problems          4880
Ankle injury               4602
Cruciate ligament tear     4035
Ill                        3694
Corona virus               3469
Knock                      2991
Name: count, dtype: int64


### `PLAYER_NATIONAL_TEAM_PERFORMANCES`

In [ ]:
print(tables["player_national_performances"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_national_performances")
pd.DataFrame({
    "nb" : tables["player_national_performances"].isna().sum(),
    "pct" : (tables["player_national_performances"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(92701, 9)
Proportion de valeurs manquantes par variables dans la table player_national_performances


,nb,pct
player_id,0,0.00
team_id,0,0.00
matches,0,0.00
goals,0,0.00
shirt_number,0,0.00
career_state,0,0.00
debut_game_id,30347,32.74
coach_id,36922,39.83
debut,92701,100.00


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_national_performances"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_national_performances"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_national_performances"].columns[tables["player_national_performances"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  92701
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index(['debut'], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.

In [ ]:
print("Doublons stricts :", tables["player_national_performances"].duplicated().sum())

Doublons stricts : 32


Les statistiques de matchs, buts, ont-elles des anomalies (ex : but negatif ?)

In [ ]:
print(
    tables["player_national_performances"][
        ["matches", "goals"]
    ].describe()
)

            matches         goals
count  92701.000000  92701.000000
mean       7.336027      0.911382
std       12.943084      2.910156
min        0.000000      0.000000
25%        1.000000      0.000000
50%        4.000000      0.000000
75%        8.000000      1.000000
max      223.000000    141.000000


### `PLAYER_TRANSFERT_HISTORIES`

In [ ]:
print(tables["transfer_history"].shape)
print("Proportion de valeurs manquantes par variables dans la table transfer_history")
pd.DataFrame({
    "nb" : tables["transfer_history"].isna().sum(),
    "pct" : (tables["transfer_history"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1101440, 10)
Proportion de valeurs manquantes par variables dans la table transfer_history


,nb,pct
player_id,0,0.00
season_name,2,0.00
from_team_id,0,0.00
from_team_name,0,0.00
to_team_id,0,0.00
to_team_name,0,0.00
transfer_type,0,0.00
value_at_transfer,0,0.00
transfer_fee,0,0.00
transfer_date,580,0.05


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["transfer_history"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["transfer_history"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["transfer_history"].columns[tables["transfer_history"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  580
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.

In [ ]:
print("Doublons stricts :", tables["transfer_history"].duplicated().sum())

Doublons stricts : 123


Est-ce que les frais de tranferts et les valeurs ont des valeurs négatives ?

In [ ]:
print(
    tables["transfer_history"][
        ["value_at_transfer", "transfer_fee"]
    ].describe()
)

       value_at_transfer  transfer_fee
count       1.101440e+06  1.101440e+06
mean        3.210798e+05  1.137681e+05
std         1.678342e+06  1.501613e+06
min         0.000000e+00  0.000000e+00
25%         0.000000e+00  0.000000e+00
50%         0.000000e+00  0.000000e+00
75%         2.000000e+05  0.000000e+00
max         1.800000e+08  2.220000e+08


### `PLAYER_MARKET_VALUES`

In [ ]:
print(tables["player_market_value"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_market_value")
pd.DataFrame({
    "nb" : tables["player_market_value"].isna().sum(),
    "pct" : (tables["player_market_value"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(901429, 3)
Proportion de valeurs manquantes par variables dans la table player_market_value


,nb,pct
player_id,0,0.0
date_unix,0,0.0
value,0,0.0


In [ ]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_market_value"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_market_value"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_market_value"].columns[tables["player_market_value"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  0
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


In [ ]:
print("Doublons stricts :", tables["player_market_value"].duplicated().sum())

Doublons stricts : 0


valeurs négatives ?

In [ ]:
print(
    tables["player_market_value"][
        ["value"]
    ].describe()
)

              value
count  9.014290e+05
mean   1.544903e+06
std    5.424065e+06
min    0.000000e+00
25%    1.000000e+05
50%    3.000000e+05
75%    8.000000e+05
max    2.000000e+08


### `PLAYER_PERFORMANCES`


In [ ]:
print(tables["player_performances"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_performances")
pd.DataFrame({
    "nb" : tables["player_performances"].isna().sum(),
    "pct" : (tables["player_performances"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1878719, 20)
Proportion de valeurs manquantes par variables dans la table player_performances


,nb,pct
player_id,0,0.00
penalty_goals,0,0.00
direct_red_cards,0,0.00
second_yellow_cards,0,0.00
yellow_cards,0,0.00
subed_out,0,0.00
subed_in,0,0.00
own_goals,0,0.00
goals_conceded,0,0.00
assists,0,0.00
